# Viscoelasticity & Creep Mechanics Calculator
**Plastic Design Calculators — Notebook 1**

---

## Part 1: Theory and Governing Equations

### 1.1 The Viscoelastic Nature of Polymers

Unlike metals, which store elastic strain energy and release it instantaneously, polymeric materials exhibit **viscoelastic** behaviour: under constant load they continue to deform over time (creep), and under constant deformation their internal stress decays over time (stress relaxation). No single Young's modulus characterises a plastic — instead, compliance and modulus are functions of both **time** and **temperature**.

### 1.2 Boltzmann Superposition Principle (BSP)

For a linear viscoelastic material subjected to a history of discrete stress increments, the **total strain at time $t$** is the sum of the strain contributions from each individual load step:

$$\boxed{\epsilon(t) = \sum_{i} \Delta\sigma_i \, J(t - t_i)}$$

where:
- $\Delta\sigma_i$ — discrete stress change applied at time $t_i$ [Pa]
- $J(t - t_i)$ — **creep compliance** function evaluated at elapsed time since load application [1/Pa]

For **continuous** loading histories, the discrete sum becomes the Boltzmann convolution integral:

$$\epsilon(t) = \int_{-\infty}^{t} J(t - \tau) \frac{d\sigma(\tau)}{d\tau} \, d\tau$$

With an instantaneous stress step $\sigma_0$ at $t = 0$ followed by a continuous history:

$$\epsilon(t) = \sigma_0 J(t) + \int_{0}^{t} J(t - \tau) \frac{d\sigma(\tau)}{d\tau} \, d\tau$$

### 1.3 Creep Compliance Power-Law Model

A practical empirical model for polymer creep compliance is:

$$J(t) = J_0 + A \, t^n$$

where $J_0 = 1/E_0$ is the instantaneous elastic compliance, and $A$, $n$ are material-specific curve-fit parameters.

### 1.4 Williams–Landel–Ferry (WLF) Time–Temperature Superposition

Because long-term testing at service temperature is impractical, short-duration tests at elevated temperature are **shifted** to predict long-term behaviour at reference conditions. The horizontal shift factor $a_T$ is computed via the WLF equation:

$$\log(a_T) = \frac{-C_1 (T - T_{\mathrm{ref}})}{C_2 + (T - T_{\mathrm{ref}})}$$

### 1.5 Effective Reduced Time

For a thermal history involving periods at different temperatures, the total **effective reduced time** at the reference temperature is:

$$t_{\mathrm{eff}} = \sum_i t_i \cdot \frac{\alpha_{T_{\mathrm{ref}}}}{\alpha_{T_i}}$$

### 1.6 Recovery After Unloading

After load removal at $t_{\mathrm{total}}$, the residual strain at recovery time $t_{\mathrm{rec}}$ is:

$$\epsilon_{\mathrm{residual}} = \sigma \cdot J(t_{\mathrm{eff,total}}) - \sigma \cdot J(t_{\mathrm{eff,rec}})$$

### Assumptions
- Linear viscoelasticity (strains < ~0.5%; stresses below yield)
- Thermorheologically simple material (WLF TTS valid)
- Uniaxial stress state
- Constant humidity (hygroscopic effects excluded)

---
## Part 2: Variable Definitions and Unit Handling

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import pint

from utils.unit_registry import ureg, Q_, strip_units, reattach_units
from utils.material_db import CREEP_COMPLIANCE, WLF_CONSTANTS, PP_SHIFT_FACTORS

# ── Material selection ────────────────────────────────────────────────────────
MATERIAL_KEY = 'PP'          # key into CREEP_COMPLIANCE database
WLF_KEY      = 'PP_23'       # key into WLF_CONSTANTS database

# ── Applied stress history ────────────────────────────────────────────────────
# Discrete BSP example: two-step loading
stress_steps = [
    {'sigma': Q_(10.0, 'MPa'),  't_apply': Q_(0.0,   'hour')},
    {'sigma': Q_( 5.0, 'MPa'),  't_apply': Q_(1000.0,'hour')},   # additional load at 1000 h
]
t_end   = Q_(10_000.0, 'hour')   # total analysis duration

# ── Thermal history for TTS reduced-time calculation ─────────────────────────
# Each entry: (duration [hours], temperature [°C])
thermal_history = [
    (Q_(4000.0, 'hour'), 23.0),   # 4000 h at 23 °C
    (Q_(4000.0, 'hour'), 40.0),   # 4000 h at 40 °C  (summer storage)
    (Q_(2000.0, 'hour'), 60.0),   # 2000 h at 60 °C  (elevated service)
]
t_recovery = Q_(500.0, 'hour')   # unloaded recovery period after full service

# ── Recover material constants ────────────────────────────────────────────────
mat  = CREEP_COMPLIANCE[MATERIAL_KEY]
wlf  = WLF_CONSTANTS[WLF_KEY]
J0   = mat['J0']   # [1/Pa] instantaneous compliance
A    = mat['A']    # power-law coefficient
n    = mat['n']    # power-law exponent
C1   = wlf['C1']
C2   = wlf['C2']
T_ref = wlf['T_ref']  # reference temperature [°C]

print(f"Material  : {mat['description']}")
print(f"J0        : {J0:.4e} 1/Pa")
print(f"A, n      : {A:.2e}, {n}")
print(f"WLF C1,C2 : {C1}, {C2}  (Tref={T_ref}°C)")

---
## Part 3: Computation Engine

In [ ]:
# ── 3.1  Symbolic representation of governing equations ──────────────────────
t_sym, tau_sym, J0_sym, A_sym, n_sym = sp.symbols('t tau J_0 A n', positive=True)
sigma_sym, Delta_sigma_sym, t_i_sym  = sp.symbols('sigma Delta_sigma t_i', positive=True)

J_sym        = J0_sym + A_sym * (t_sym - tau_sym)**n_sym
bsp_discrete = sp.Sum(Delta_sigma_sym * (J0_sym + A_sym*(t_sym - t_i_sym)**n_sym),
                      (t_i_sym, 0, t_sym))
wlf_eq       = -sp.Symbol('C_1')*(t_sym - sp.Symbol('T_ref')) / \
               (sp.Symbol('C_2') + (t_sym - sp.Symbol('T_ref')))

print("Creep compliance J(t-τ) =")
sp.pprint(J_sym)
print()
print("BSP discrete sum ε(t) =")
sp.pprint(bsp_discrete)

In [ ]:
def creep_compliance(elapsed_time_s, J0, A, n, **kwargs):
    """Power-law creep compliance function J(t) = J0 + A*t^n.

    Args:
        elapsed_time_s (numpy.ndarray | float): Elapsed time since load application [s].
        J0 (float): Instantaneous elastic compliance [1/Pa].
        A (float): Power-law pre-factor [1/(Pa·s^n)].
        n (float): Power-law exponent [-].
        **kwargs: Reserved for extended compliance models (e.g. hygroscopic offset).

    Returns:
        numpy.ndarray | float: Creep compliance [1/Pa].
    """
    elapsed = np.maximum(elapsed_time_s, 0.0)   # compliance undefined for t < 0
    return J0 + A * elapsed**n


def boltzmann_discrete(t_eval_s, stress_steps_Pa, t_apply_s, J0, A, n, **kwargs):
    """Discrete Boltzmann Superposition: ε(t) = Σ Δσᵢ · J(t - tᵢ).

    Args:
        t_eval_s (numpy.ndarray): Time points for evaluation [s].
        stress_steps_Pa (list[float]): Applied stress increments Δσᵢ [Pa].
        t_apply_s (list[float]): Corresponding application times tᵢ [s].
        J0, A, n: Creep compliance parameters.
        **kwargs: Forwarded to creep_compliance.

    Returns:
        numpy.ndarray: Total strain ε(t) [-] at each evaluation point.
    """
    epsilon = np.zeros_like(t_eval_s, dtype=float)
    for delta_sigma, t_apply in zip(stress_steps_Pa, t_apply_s):
        # vectorised: only accumulate where t >= t_apply
        active = t_eval_s >= t_apply
        elapsed = np.where(active, t_eval_s - t_apply, 0.0)
        epsilon += delta_sigma * creep_compliance(elapsed, J0, A, n, **kwargs) * active
    return epsilon


def wlf_shift_factor(T_celsius, T_ref, C1, C2, **kwargs):
    """Williams-Landel-Ferry horizontal shift factor log(aT).

    Args:
        T_celsius (float | numpy.ndarray): Operating temperature [°C].
        T_ref (float): Reference temperature [°C].
        C1 (float): WLF constant 1.
        C2 (float): WLF constant 2.
        **kwargs: Reserved for pressure-dependent WLF extensions.

    Returns:
        float | numpy.ndarray: Shift factor aT (not log).
    """
    delta_T = np.asarray(T_celsius) - T_ref
    log_aT  = -C1 * delta_T / (C2 + delta_T)
    return 10.0**log_aT


def reduced_time(thermal_history_list, shift_factor_at_T_ref, C1, C2, T_ref, **kwargs):
    """Compute effective reduced time from a multi-temperature thermal history.

    Args:
        thermal_history_list (list[tuple]): [(duration_s, temperature_C), ...].
        shift_factor_at_T_ref (float): α_Tref value (shift factor at reference temperature).
        C1, C2, T_ref: WLF parameters.
        **kwargs: Reserved for UV-degradation time acceleration factors.

    Returns:
        float: Total effective reduced time [s].
    """
    t_eff = 0.0
    for duration_s, temp_C in thermal_history_list:
        alpha_Ti = wlf_shift_factor(temp_C, T_ref, C1, C2)
        t_eff   += duration_s * (shift_factor_at_T_ref / alpha_Ti)
    return t_eff


print("Computation functions defined.")

In [ ]:
# ── 3.2  Numerical execution ─────────────────────────────────────────────────

# Convert all inputs to SI base units (seconds, Pascals) before computation
t_end_s   = strip_units(t_end.to('second'))
steps_Pa  = [strip_units(s['sigma'].to('Pa'))    for s in stress_steps]
apply_s   = [strip_units(s['t_apply'].to('second')) for s in stress_steps]

# Log-spaced time vector: 1 second → t_end
t_arr = np.logspace(0, np.log10(t_end_s), 500)   # 500 points, vectorised

# Discrete BSP strain profile
epsilon_bsp = boltzmann_discrete(t_arr, steps_Pa, apply_s, J0, A, n)

# Thermal history reduced time
history_si = [(strip_units(d.to('second')), T) for d, T in thermal_history]
alpha_ref  = PP_SHIFT_FACTORS.get(int(T_ref), wlf_shift_factor(T_ref, T_ref, C1, C2))
# Note: wlf_shift_factor(T_ref, T_ref, ...) = 1.0; use tabulated value if available
t_eff_s    = reduced_time(history_si, alpha_ref, C1, C2, T_ref)
t_rec_s    = strip_units(t_recovery.to('second'))

# Residual strain after recovery
sigma_total_Pa   = sum(steps_Pa)
epsilon_residual = sigma_total_Pa * (creep_compliance(t_eff_s, J0, A, n)
                                   - creep_compliance(t_rec_s,   J0, A, n))

t_eff_h = t_eff_s / 3600.0
print(f"Effective reduced service time : {t_eff_h:,.1f} hours")
print(f"Total strain at t_eff          : {sigma_total_Pa * creep_compliance(t_eff_s,J0,A,n)*100:.4f} %")
print(f"Residual strain after recovery : {epsilon_residual*100:.4f} %")

---
## Part 4: Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Strain vs log-time (BSP discrete) ────────────────────────────────
ax1 = axes[0]
ax1.semilogx(t_arr / 3600.0, epsilon_bsp * 100, color='steelblue', lw=2)
for step in stress_steps:
    t_mark = strip_units(step['t_apply'].to('hour'))
    if t_mark > 0:
        ax1.axvline(t_mark, color='tomato', ls='--', lw=1, label=f'Load step at {t_mark:.0f} h')
ax1.set_xlabel('Time [hours]')
ax1.set_ylabel('Total Strain ε [%]')
ax1.set_title(f'Creep Strain — BSP Discrete\n({mat["description"]})')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)

# ── Plot 2: WLF shift factor vs temperature ──────────────────────────────────
T_range   = np.linspace(T_ref, T_ref + 80, 200)
aT_range  = wlf_shift_factor(T_range, T_ref, C1, C2)

ax2 = axes[1]
ax2.semilogy(T_range, aT_range, color='darkorange', lw=2)
ax2.set_xlabel('Temperature [°C]')
ax2.set_ylabel('Shift factor $a_T$  [—]')
ax2.set_title(f'WLF Shift Factor\n(Tref={T_ref}°C, C1={C1}, C2={C2})')
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('01_creep_output.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved → 01_creep_output.png")

---
## Part 5: Design Rule Validation

In [ ]:
# ── Design limits ─────────────────────────────────────────────────────────────
MAX_ALLOWABLE_STRAIN_PCT  = 1.5    # typical short-term strain limit for PP structural parts
MAX_RESIDUAL_STRAIN_PCT   = 0.3    # allowable permanent set after recovery

strain_at_t_eff_pct  = sigma_total_Pa * creep_compliance(t_eff_s, J0, A, n) * 100.0
residual_strain_pct  = epsilon_residual * 100.0

pass_creep    = strain_at_t_eff_pct  <= MAX_ALLOWABLE_STRAIN_PCT
pass_recovery = residual_strain_pct  <= MAX_RESIDUAL_STRAIN_PCT

def result_badge(passed):
    return "\033[92m  PASS  \033[0m" if passed else "\033[91m  FAIL  \033[0m"

print("═" * 60)
print("  DESIGN RULE VALIDATION — VISCOELASTICITY & CREEP")
print("═" * 60)
print(f"  Creep strain at effective service life:")
print(f"    Computed : {strain_at_t_eff_pct:.3f} %")
print(f"    Limit    : {MAX_ALLOWABLE_STRAIN_PCT:.3f} %")
print(f"    Result   : {result_badge(pass_creep)}")
print()
print(f"  Residual strain after {t_recovery} recovery:")
print(f"    Computed : {residual_strain_pct:.4f} %")
print(f"    Limit    : {MAX_RESIDUAL_STRAIN_PCT:.3f} %")
print(f"    Result   : {result_badge(pass_recovery)}")
print("═" * 60)

overall = pass_creep and pass_recovery
if overall:
    print("  ✓ OVERALL: DESIGN PASSES creep and recovery criteria.")
else:
    print("  ✗ OVERALL: DESIGN FAILS — review geometry or material.")
print("═" * 60)